# 🏦 Trade Surveillance — Behavioral Profile Drift
### Adapted from IEEE-CIS Fraud Detection Competition

**Concept:** Reframe IEEE-CIS transaction fraud detection → **trade surveillance & behavioral drift monitoring**

| IEEE-CIS Original | Trade Surveillance Context |
|---|---|
| `TransactionAmt` | `order_notional` (trade size) |
| `card1 / card2` | `trader_id / desk_id` |
| `D1–D15` (time deltas) | `settlement_lag / order_lifetime` |
| `C1–C14` (counts) | `order_count / cancel_count` |
| `isFraud` | `is_suspicious` (surveillance flag) |
| `ProductCD` | `instrument_type` (equity, FX, derivative) |

---
## 📁 Data Upload Instructions
1. Download from: `kaggle.com/competitions/ieee-fraud-detection` → Data tab
2. Files needed: **`train_transaction.csv`** and **`train_identity.csv`**
3. Upload via the cell below (or drag-drop to Colab left panel → Files)

In [ ]:
# ============================================================
# CELL 1 — Install & Import
# ============================================================
!pip install -q plotly seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import os, warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded')

In [ ]:
# ============================================================
# CELL 2 — Upload CSV Files Manually
# ============================================================
from google.colab import files

print('📁 Upload train_transaction.csv and train_identity.csv')
print('   (both files at once, Ctrl+click to select multiple)')
uploaded = files.upload()

# Verify both files present
required = ['train_transaction.csv', 'train_identity.csv']
for f in required:
    if f in uploaded:
        size_mb = len(uploaded[f]) / 1024 / 1024
        print(f'  ✅ {f} ({size_mb:.1f} MB)')
    else:
        print(f'  ❌ {f} NOT found — please re-upload')

In [ ]:
# ============================================================
# CELL 3 — Load Data & Merge
# ============================================================
import io

print('📊 Loading transaction data...')
trans    = pd.read_csv(io.BytesIO(uploaded['train_transaction.csv']))
identity = pd.read_csv(io.BytesIO(uploaded['train_identity.csv']))

print(f'  Transaction shape : {trans.shape}')
print(f'  Identity shape    : {identity.shape}')

df = trans.merge(identity, on='TransactionID', how='left')
print(f'  Merged shape      : {df.shape}')
print(f'  Fraud rate        : {df["isFraud"].mean():.2%}')
print('✅ Data loaded successfully')

In [ ]:
# ============================================================
# CELL 4 — Column Remapping (IEEE → Trade Surveillance)
# ============================================================
df_surv = df.copy()

df_surv['order_notional']    = df_surv['TransactionAmt']
df_surv['trader_id']         = df_surv['card1'].astype(str)
df_surv['desk_id']           = df_surv['card2'].astype(str)
df_surv['order_lifetime_d1'] = df_surv['D1']
df_surv['order_lifetime_d2'] = df_surv['D2']
df_surv['cancel_count']      = df_surv['C1']
df_surv['order_count']       = df_surv['C2']
df_surv['instrument_type']   = df_surv['ProductCD']
df_surv['is_suspicious']     = df_surv['isFraud']

df_surv['order_type'] = np.where(
    df_surv['order_notional'] > df_surv['order_notional'].median(), 'BUY', 'SELL'
)
df_surv['cancel_rate'] = df_surv['cancel_count'] / (df_surv['order_count'] + 1e-8)
df_surv['notional_bucket'] = pd.qcut(
    df_surv['order_notional'], q=5,
    labels=['XS','S','M','L','XL']
)

print('✅ Column remapping complete')
df_surv[['trader_id','order_notional','cancel_rate','instrument_type','is_suspicious']].head(5)

## Step 3 — Exploratory Data Analysis

In [ ]:
# ============================================================
# CELL 5 — EDA Overview (4 charts)
# ============================================================
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Order Notional Distribution (Log Scale)',
        'Suspicious vs Normal — Notional Box',
        'Cancel Rate by Instrument Type',
        'Order Count Distribution'
    ]
)

fig.add_trace(go.Histogram(
    x=np.log1p(df_surv['order_notional']),
    nbinsx=60, name='Log Notional', marker_color='#2196F3'
), row=1, col=1)

for label, color in [(0, '#4CAF50'), (1, '#F44336')]:
    subset = df_surv[df_surv['is_suspicious'] == label]
    fig.add_trace(go.Box(
        y=np.log1p(subset['order_notional']),
        name='Suspicious' if label==1 else 'Normal',
        marker_color=color
    ), row=1, col=2)

cancel_by_inst = df_surv.groupby('instrument_type')['cancel_rate'].mean().reset_index()
fig.add_trace(go.Bar(
    x=cancel_by_inst['instrument_type'],
    y=cancel_by_inst['cancel_rate'],
    name='Avg Cancel Rate', marker_color='#FF9800'
), row=2, col=1)

fig.add_trace(go.Histogram(
    x=df_surv['order_count'].clip(0, 50),
    nbinsx=50, name='Order Count', marker_color='#9C27B0'
), row=2, col=2)

fig.update_layout(height=700, title_text='📊 Trade Surveillance — EDA Overview', showlegend=True)
fig.show()

## Step 4 — Feature Engineering (Behavioral Profile)

In [ ]:
# ============================================================
# CELL 6 — Feature Engineering
# ============================================================
def engineer_features(df):
    df = df.copy()

    # Notional velocity: order size vs trader's own median
    trader_median = df.groupby('trader_id')['order_notional'].transform('median')
    df['notional_velocity'] = df['order_notional'] / (trader_median + 1e-8)

    # Cancel aggression
    df['cancel_aggression'] = df['cancel_count'] / (df['order_count'] + 1e-8)

    # Desk z-score: how unusual vs desk peers
    desk_mean = df.groupby('desk_id')['order_notional'].transform('mean')
    desk_std  = df.groupby('desk_id')['order_notional'].transform('std').fillna(1)
    df['desk_notional_zscore'] = (df['order_notional'] - desk_mean) / (desk_std + 1e-8)

    # Settlement lag
    df['settlement_lag']  = df['order_lifetime_d1'].fillna(0)
    df['lag_acceleration'] = df['order_lifetime_d2'].fillna(0) - df['order_lifetime_d1'].fillna(0)

    # Large order flag (top 5% per instrument)
    df['is_large_order'] = (
        df.groupby('instrument_type')['order_notional']
        .transform(lambda x: x > x.quantile(0.95))
    ).astype(int)

    # Trader-level aggregates
    trader_agg = df.groupby('trader_id').agg(
        trader_avg_notional = ('order_notional', 'mean'),
        trader_std_notional = ('order_notional', 'std'),
        trader_cancel_rate  = ('cancel_aggression', 'mean'),
        trader_large_orders = ('is_large_order', 'sum'),
        trader_order_count  = ('order_count', 'sum')
    ).reset_index()

    df = df.merge(trader_agg, on='trader_id', how='left')

    # Coefficient of variation (erratic behavior)
    df['notional_cov'] = df['trader_std_notional'] / (df['trader_avg_notional'] + 1e-8)

    return df

print('⚙️  Engineering behavioral features...')
df_feat = engineer_features(df_surv)
print(f'✅ Done — shape: {df_feat.shape}')

In [ ]:
# ============================================================
# CELL 7 — Prepare Model Input
# ============================================================
feature_cols = [
    'order_notional', 'cancel_aggression', 'notional_velocity',
    'desk_notional_zscore', 'settlement_lag', 'lag_acceleration',
    'is_large_order', 'trader_avg_notional', 'trader_std_notional',
    'trader_cancel_rate', 'trader_large_orders', 'trader_order_count',
    'notional_cov', 'order_count', 'cancel_count'
]

model_df = df_feat[feature_cols + ['is_suspicious', 'trader_id']].copy()
model_df = model_df.fillna(model_df.median(numeric_only=True))

X = model_df[feature_cols].values
y = model_df['is_suspicious'].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'✅ Model input ready — {X_scaled.shape[0]:,} orders × {X_scaled.shape[1]} features')
print(f'   Suspicious orders : {y.sum():,} ({y.mean():.2%})')

## Step 5 — Isolation Forest

In [ ]:
# ============================================================
# CELL 8 — Train Isolation Forest
# ============================================================
contamination = float(y.mean())

print(f'🌲 Training Isolation Forest (contamination={contamination:.3f})...')
model = IsolationForest(
    n_estimators=300,
    contamination=contamination,
    max_samples=0.8,
    random_state=42,
    n_jobs=-1
)
model.fit(X_scaled)

model_df = model_df.copy()
model_df['anomaly_flag'] = model.predict(X_scaled)
model_df['anomaly_flag'] = model_df['anomaly_flag'].map({1: 0, -1: 1})

raw_scores = -model.score_samples(X_scaled)
model_df['anomaly_prob'] = (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())

print('✅ Model trained')
print(f'   Flagged as anomaly : {model_df["anomaly_flag"].sum():,}')

## Step 6 — Evaluation

In [ ]:
# ============================================================
# CELL 9 — Evaluation Metrics
# ============================================================
print('=' * 55)
print('  TRADE SURVEILLANCE — MODEL EVALUATION REPORT')
print('=' * 55)
print(classification_report(
    model_df['is_suspicious'],
    model_df['anomaly_flag'],
    target_names=['Normal', 'Suspicious']
))

auc = roc_auc_score(model_df['is_suspicious'], model_df['anomaly_prob'])
print(f'🎯 ROC AUC Score: {auc:.4f}')

cm = confusion_matrix(model_df['is_suspicious'], model_df['anomaly_flag'])
fig_cm, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Suspicious'],
            yticklabels=['Normal','Suspicious'])
ax.set_title('Confusion Matrix — Trade Surveillance')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

## Step 7 — Surveillance Dashboard

In [ ]:
# ============================================================
# CELL 10 — Chart 1: Top Flagged Traders
# ============================================================
trader_risk = model_df.groupby('trader_id').agg(
    max_anomaly_prob   = ('anomaly_prob', 'max'),
    avg_notional       = ('order_notional', 'mean'),
    avg_cancel_rate    = ('cancel_aggression', 'mean'),
    flagged_orders     = ('anomaly_flag', 'sum'),
    is_suspicious_true = ('is_suspicious', 'max')
).reset_index().sort_values('max_anomaly_prob', ascending=False).head(20)

trader_risk['label'] = trader_risk['is_suspicious_true'].map(
    {1: '🔴 Confirmed', 0: '🟡 Under Review'}
)

fig1 = px.bar(
    trader_risk,
    x='trader_id', y='max_anomaly_prob',
    color='label',
    color_discrete_map={'🔴 Confirmed': '#F44336', '🟡 Under Review': '#FF9800'},
    title='🚨 Top 20 Highest-Risk Traders — Anomaly Probability Score',
    labels={'max_anomaly_prob': 'Risk Score', 'trader_id': 'Trader ID'}
)
fig1.add_hline(y=0.7, line_dash='dash', line_color='red',
               annotation_text='Alert Threshold (0.70)')
fig1.update_layout(height=450, xaxis_tickangle=45)
fig1.show()

In [ ]:
# ============================================================
# CELL 11 — Chart 2: Cancel Aggression vs Anomaly Probability
# ============================================================
sample = model_df.sample(min(5000, len(model_df)), random_state=42)

fig2 = px.scatter(
    sample,
    x='cancel_aggression', y='anomaly_prob',
    color='is_suspicious',
    color_discrete_map={0: '#4CAF50', 1: '#F44336'},
    size='order_notional', size_max=12, opacity=0.6,
    title='📈 Cancel Aggression vs Anomaly Probability',
    labels={
        'cancel_aggression': 'Cancel Rate',
        'anomaly_prob': 'Anomaly Score',
        'is_suspicious': 'Ground Truth'
    }
)
fig2.add_hline(y=0.7, line_dash='dash', line_color='orange',
               annotation_text='Alert Threshold')
fig2.update_layout(height=500)
fig2.show()

In [ ]:
# ============================================================
# CELL 12 — Chart 3: Notional Velocity Distribution
# ============================================================
fig3 = go.Figure()
for label, color, name in [(0, '#4CAF50', 'Normal'), (1, '#F44336', 'Suspicious')]:
    subset = model_df[model_df['is_suspicious'] == label]['notional_velocity'].clip(0, 10)
    fig3.add_trace(go.Histogram(
        x=subset, nbinsx=80, name=name, opacity=0.65, marker_color=color
    ))
fig3.update_layout(
    barmode='overlay',
    title='🔍 Order Notional Velocity — Normal vs Suspicious',
    xaxis_title='Notional Velocity (order / trader median)',
    yaxis_title='Count', height=450
)
fig3.show()

In [ ]:
# ============================================================
# CELL 13 — Chart 4: Feature Importance
# ============================================================
importance = pd.Series(
    [abs(model_df[col].corr(model_df['anomaly_prob'])) for col in feature_cols],
    index=feature_cols
).sort_values(ascending=True).tail(12)

fig4 = go.Figure(go.Bar(
    x=importance.values, y=importance.index,
    orientation='h',
    marker=dict(color=importance.values, colorscale='RdYlGn_r')
))
fig4.update_layout(
    title='🧠 Feature Importance — Correlation with Anomaly Score',
    xaxis_title='Absolute Correlation', height=450
)
fig4.show()

In [ ]:
# ============================================================
# CELL 14 — Operations Summary Report
# ============================================================
high_risk      = model_df[model_df['anomaly_prob'] > 0.7]
confirmed_hits = model_df[(model_df['anomaly_prob'] > 0.7) & (model_df['is_suspicious'] == 1)]
total_suspicious = int(model_df['is_suspicious'].sum())

print('=' * 60)
print('       TRADE SURVEILLANCE — OPERATIONS SUMMARY REPORT')
print('=' * 60)
print(f'  Total orders monitored       : {len(model_df):>12,}')
print(f'  Unique traders tracked       : {model_df["trader_id"].nunique():>12,}')
print(f'  Orders flagged (score > 0.7) : {len(high_risk):>12,}')
print(f'  Confirmed suspicious orders  : {total_suspicious:>12,}')
print(f'  True positives detected      : {len(confirmed_hits):>12,}')
print(f'  Detection rate               : {len(confirmed_hits)/total_suspicious:.2%}')
print(f'  ROC AUC                      : {auc:>12.4f}')
print('=' * 60)
print('\n🔴 TOP 5 HIGHEST RISK TRADERS:')
print(trader_risk[['trader_id','max_anomaly_prob','avg_cancel_rate','flagged_orders','label']]
      .head(5).to_string(index=False))
print('\n✅ Report generated.')